<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/BeatLab_AI_Artifact_Remover_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎛️ BeatLab — AI Audio Artifact Detector & Remover
> **Google Colab Tool** — Companion to the Beat Lab web app  
> Upload a WAV or MP3, detect AI-generation signatures, and export a processed version with artifacts reduced.

---
### What this tool does
| Stage | What happens |
|---|---|
| **1. Analysis** | Measures 6 signal characteristics known to differ between AI-generated and human-produced audio |
| **2. Scoring** | Produces a suspicion score (0–100%) with per-metric breakdown |
| **3. Auto-Correct** | Reads every score, derives optimal correction strength per artifact, and applies the full pipeline automatically — zero config needed |
| **4. Manual mode** | Optional manual config cells let you fine-tune individual stages after autocorrect |
| **5. Export** | Downloads the processed audio as a 44.1 kHz 24-bit WAV |

### Limitations
- This is **heuristic** analysis, not a neural classifier. It can produce false positives on heavily mastered or heavily quantised human productions.
- Processing improves naturalness; it cannot "un-generate" AI content.
- For professional use, treat the output as a starting point for further mixing.


In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────
!pip install -q librosa soundfile noisereduce matplotlib numpy scipy

In [ ]:
# ── Cell 2: Imports ─────────────────────────────────────────────────
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal
from scipy.stats import variation
from google.colab import files
import io, os, warnings
warnings.filterwarnings('ignore')

print('✅ All dependencies loaded')

In [ ]:
# ── Cell 3: Upload your audio file ──────────────────────────────────
print('📂 Select a WAV or MP3 file to analyse...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
audio_bytes = uploaded[fname]

y, sr = librosa.load(io.BytesIO(audio_bytes), sr=None, mono=False)
if y.ndim == 1:
    y = np.stack([y, y])
y_mono = librosa.to_mono(y)
duration = len(y_mono) / sr

print(f'\n✅ Loaded: {fname}')
print(f'   Sample rate : {sr} Hz')
print(f'   Duration    : {duration:.2f} s')
print(f'   Channels    : {y.shape[0]}')
print(f'   Samples     : {y.shape[1]:,}')

In [ ]:
# ── Cell 4: AI Artifact Analysis ────────────────────────────────────
results = {}

# 1 — Dynamic Range
peak = np.max(np.abs(y_mono))
rms  = np.sqrt(np.mean(y_mono**2))
crest_db = 20*np.log10(peak/rms) if rms>0 else 20
dr_score = max(0, min(1, (16-crest_db)/9))
results['Dynamic Range'] = {'value':f'{crest_db:.1f} dB','score':dr_score,'raw':crest_db,
    'detail':'Over-limited (AI)' if crest_db<8 else 'Compressed' if crest_db<12 else 'Natural'}

# 2 — Spectral Flatness
flatness = librosa.feature.spectral_flatness(y=y_mono)
mean_flat = float(np.mean(flatness))
sf_score = (max(0,min(1,(mean_flat-0.25)/0.3)) if mean_flat>0.25
            else max(0,min(1,(0.03-mean_flat)/0.025)) if mean_flat<0.03 else 0)
results['Spectral Flatness'] = {'value':f'{mean_flat:.4f}','score':sf_score,'raw':mean_flat,
    'detail':'Unnaturally flat' if mean_flat>0.25 else 'Over-processed' if mean_flat<0.03 else 'Normal'}

# 3 — Timing Regularity
tempo, beat_frames = librosa.beat.beat_track(y=y_mono, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)
if len(beat_times)>4:
    ibi=np.diff(beat_times)*1000; jitter_ms=float(np.std(ibi))
    tr_score=max(0,min(1,(8-jitter_ms)/7))
else:
    jitter_ms=999; tr_score=0.5
results['Timing Regularity'] = {'value':f'{jitter_ms:.1f} ms' if jitter_ms<900 else 'N/A',
    'score':tr_score,'raw':jitter_ms,
    'detail':'Machine-perfect' if jitter_ms<3 else 'Tight' if jitter_ms<8 else 'Natural jitter'}

# 4 — Stereo Width
mid=(y[0]+y[1])*0.5; side=(y[0]-y[1])*0.5
ms_ratio=np.mean(side**2)/(np.mean(mid**2)+1e-10)
sw_score=(max(0,min(1,(ms_ratio-0.75)/0.4)) if ms_ratio>0.75
          else max(0,min(1,(0.03-ms_ratio)/0.025)) if ms_ratio<0.03 else 0)
results['Stereo Width'] = {'value':f'{ms_ratio:.3f}','score':sw_score,'raw':ms_ratio,
    'detail':'Too wide' if ms_ratio>0.75 else 'Near-mono' if ms_ratio<0.03 else 'Balanced'}

# 5 — HF Energy
S=np.abs(librosa.stft(y_mono)); freqs=librosa.fft_frequencies(sr=sr)
hf_mask=freqs>15000
hf_ratio=float(np.mean(S[hf_mask,:]**2)/(np.mean(S**2)+1e-10))
hf_score=(max(0,min(1,(0.002-hf_ratio)/0.002)) if hf_ratio<0.002
          else max(0,min(1,(hf_ratio-0.07)/0.06)) if hf_ratio>0.07 else 0)
results['HF Energy'] = {'value':f'{hf_ratio*100:.3f}%','score':hf_score,'raw':hf_ratio,
    'detail':'HF dead' if hf_ratio<0.002 else 'HF boosted' if hf_ratio>0.07 else 'Normal'}

# 6 — Noise Floor
rms_frames=librosa.feature.rms(y=y_mono,frame_length=2048,hop_length=1024)[0]
db_floor=20*np.log10(np.percentile(rms_frames,5)+1e-10)
nf_score=max(0,min(1,(-db_floor-80)/30))
results['Noise Floor'] = {'value':f'{db_floor:.1f} dBFS','score':nf_score,'raw':db_floor,
    'detail':'Suspiciously clean' if db_floor<-90 else 'Very quiet' if db_floor<-75 else 'Normal'}

# Combined
weights={'Dynamic Range':0.25,'Spectral Flatness':0.18,'Timing Regularity':0.22,
         'Stereo Width':0.10,'HF Energy':0.12,'Noise Floor':0.13}
combined=sum(results[k]['score']*w for k,w in weights.items())
verdict=('🔴 LIKELY AI-GENERATED' if combined>0.65
         else '🟡 UNCERTAIN / AMBIGUOUS' if combined>0.38 else '🟢 LIKELY HUMAN-PRODUCED')

print('\n'+'═'*55)
print(f'  AI ARTIFACT ANALYSIS — {fname}')
print('═'*55)
for name,r in results.items():
    bar='█'*int(r['score']*20)+'░'*(20-int(r['score']*20))
    print(f'  {name:<22} {bar}  {r["value"]:>15}  {r["detail"]}')
print('─'*55)
print(f'  SUSPICION SCORE    {combined*100:>5.1f}%')
print(f'  VERDICT            {verdict}')
print('═'*55)

In [ ]:
# ── Cell 5: Visualisation Dashboard ────────────────────────────────
fig=plt.figure(figsize=(16,12),facecolor='#07070c')
fig.suptitle(f'Beat Lab — AI Signal Analysis\n{fname}',color='#f59e0b',fontsize=14,fontweight='bold',y=0.98)
gs=gridspec.GridSpec(3,2,figure=fig,hspace=0.45,wspace=0.35)

def style_ax(ax,title):
    ax.set_facecolor('#111118'); ax.set_title(title,color='#f0eff8',fontsize=10,pad=8)
    for sp in ax.spines.values(): sp.set_color('#22223a')
    ax.tick_params(colors='#6868a0',labelsize=8)
    ax.xaxis.label.set_color('#6868a0'); ax.yaxis.label.set_color('#6868a0')

ax1=fig.add_subplot(gs[0,:])
ax1.plot(np.linspace(0,duration,len(y_mono)),y_mono,color='#f59e0b',lw=0.4,alpha=0.85)
for bt in beat_times: ax1.axvline(bt,color='#f87171',alpha=0.5,lw=0.8)
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
style_ax(ax1,'Waveform + Beat Markers')

ax2=fig.add_subplot(gs[1,0])
D=librosa.amplitude_to_db(np.abs(librosa.stft(y_mono)),ref=np.max)
librosa.display.specshow(D,sr=sr,x_axis='time',y_axis='hz',ax=ax2,cmap='magma')
style_ax(ax2,'Spectrogram')

ax3=fig.add_subplot(gs[1,1])
ft=librosa.frames_to_time(np.arange(len(flatness[0])),sr=sr)
ax3.plot(ft,flatness[0],color='#7dd3fc',lw=0.7)
ax3.axhline(0.25,color='#f87171',ls='--',lw=1,label='AI threshold')
ax3.set_xlabel('Time (s)'); ax3.set_ylabel('Flatness')
ax3.legend(fontsize=7,labelcolor='#9090c0',facecolor='#111118',edgecolor='#22223a')
style_ax(ax3,'Spectral Flatness')

ax4=fig.add_subplot(gs[2,0])
names=[n for n in results]; scores_list=[results[n]['score'] for n in names]
colors_list=['#f87171' if s>0.65 else '#f59e0b' if s>0.38 else '#4ade80' for s in scores_list]
bars=ax4.barh(names,scores_list,color=colors_list,height=0.55)
ax4.axvline(0.65,color='#f87171',ls=':',lw=1,alpha=0.7)
ax4.axvline(0.38,color='#f59e0b',ls=':',lw=1,alpha=0.7)
ax4.set_xlim(0,1); ax4.set_xlabel('Suspicion Score')
for bar,s in zip(bars,scores_list):
    ax4.text(s+0.02,bar.get_y()+bar.get_height()/2,f'{s*100:.0f}%',va='center',ha='left',color='#f0eff8',fontsize=8)
style_ax(ax4,'Per-Metric Suspicion Scores')

ax5=fig.add_subplot(gs[2,1])
theta=np.linspace(np.pi,0,300)
ax5.plot(np.cos(theta),np.sin(theta),color='#22223a',lw=12,solid_capstyle='round')
gc='#f87171' if combined>0.65 else '#f59e0b' if combined>0.38 else '#4ade80'
ts=np.linspace(np.pi,np.pi-combined*np.pi,300)
ax5.plot(np.cos(ts),np.sin(ts),color=gc,lw=12,solid_capstyle='round')
ax5.text(0,0.15,f'{combined*100:.0f}%',ha='center',va='center',color=gc,fontsize=28,fontweight='bold')
vs='LIKELY AI' if combined>0.65 else 'UNCERTAIN' if combined>0.38 else 'LIKELY HUMAN'
ax5.text(0,-0.25,vs,ha='center',va='center',color=gc,fontsize=11,fontweight='bold')
ax5.set_xlim(-1.3,1.3); ax5.set_ylim(-0.5,1.2); ax5.axis('off')
ax5.set_facecolor('#111118'); ax5.set_title('Combined Score',color='#f0eff8',fontsize=10,pad=8)

plt.savefig('beatlab_ai_analysis.png',dpi=150,bbox_inches='tight',facecolor='#07070c',edgecolor='none')
plt.show()
print('📊 Dashboard saved.')

---
## ⚡ Auto-Correct Mode
**Run Cell 6-AUTO instead of (or before) the manual config below.**  
It reads every metric score from the analysis, computes the optimal correction strength for each artifact proportionally, applies the full DSP pipeline, shows a before/after comparison table, and exports the fixed WAV — **zero configuration needed**.  
After it finishes you can still use the manual config cells (6–9) to tweak further.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ── Cell 6-AUTO: Autocorrect ──────────────────────────────────────────
# Reads analysis scores → derives correction strengths → applies all
# stages → shows before/after table → exports fixed WAV automatically.
# ══════════════════════════════════════════════════════════════════════

from scipy.signal import resample
print('═'*58)
print('  BEATLAB AUTOCORRECT')
print('═'*58)

# ── Step 1: Derive correction strengths from scores ──────────────────
def strength(metric_key, min_trigger=0.20, max_str=1.0):
    s = results[metric_key]['score']
    if s < min_trigger: return 0.0
    return float(np.clip((s - min_trigger) / (1.0 - min_trigger) * max_str, 0, max_str))

AUTO_CFG = {
    'dr_expansion'    : strength('Dynamic Range',     min_trigger=0.20, max_str=1.8),
    'spectral_smooth' : strength('Spectral Flatness', min_trigger=0.18, max_str=2.5),
    'humanise_timing' : strength('Timing Regularity', min_trigger=0.22, max_str=1.5),
    'stereo_correct'  : strength('Stereo Width',      min_trigger=0.20, max_str=1.0),
    'hf_restore_db'   : strength('HF Energy',         min_trigger=0.25, max_str=1.0) * 4.0,
    'noise_floor_db'  : -90.0 if results['Noise Floor']['score'] > 0.25 else 0.0,
    'output_target'   : -1.0,
}

print('\n  Derived correction plan:')
print(f'  {"Metric":<26} {"Score":>7}   {"Strength":>9}   Action')
print('  '+'─'*66)
plan = [('Dynamic Range','dr_expansion','Expand dynamics'),
        ('Spectral Flatness','spectral_smooth','Smooth spectrum'),
        ('Timing Regularity','humanise_timing','Add timing jitter'),
        ('Stereo Width','stereo_correct','Narrow stereo field'),
        ('HF Energy','hf_restore_db','Restore HF shelf'),
        ('Noise Floor','noise_floor_db','Inject noise floor')]
for metric,key,action in plan:
    sc=results[metric]['score']*100; val=AUTO_CFG[key]
    active=val!=0.0
    print(f'  {"✓" if active else "–"} {metric:<24} {sc:>6.1f}%   {val:>9.3f}   {action if active else "skipped"}')
print()

# ── Step 2: Apply pipeline ────────────────────────────────────────────
y_ac = y.copy().astype(np.float32)
ac_log = []

# 2a Dynamic Range Expansion
if AUTO_CFG['dr_expansion'] > 0 and results['Dynamic Range']['raw'] < 14:
    rms_ac=np.sqrt(np.mean(y_ac[0]**2)); thr=rms_ac*0.35
    for ch in range(y_ac.shape[0]):
        mask=np.abs(y_ac[ch])<thr
        y_ac[ch][mask]*=1-(AUTO_CFG['dr_expansion']*0.12*(1-np.abs(y_ac[ch][mask])/(thr+1e-10)))
    ac_log.append(f'✓ DR expansion         strength={AUTO_CFG["dr_expansion"]:.3f}')
else: ac_log.append('– DR expansion         skipped')

# 2b Spectral Smoothing
if AUTO_CFG['spectral_smooth'] > 0:
    for ch in range(y_ac.shape[0]):
        y_ac[ch]=nr.reduce_noise(y=y_ac[ch],y_noise=y_ac[ch][:sr],sr=sr,
            prop_decrease=min(0.7,0.25*AUTO_CFG['spectral_smooth']),
            freq_mask_smooth_hz=500,time_mask_smooth_ms=100).astype(np.float32)
    ac_log.append(f'✓ Spectral smoothing   strength={AUTO_CFG["spectral_smooth"]:.3f}')
else: ac_log.append('– Spectral smoothing   skipped')

# 2c Humanise Timing
if AUTO_CFG['humanise_timing'] > 0 and len(beat_times) > 2:
    jmax=AUTO_CFG['humanise_timing']*8.0/1000.0
    np.random.seed(42)
    anch=np.concatenate([[0],beat_times,[y_ac.shape[1]/sr]])
    offs=np.concatenate([[0],np.random.uniform(-jmax,jmax,len(beat_times)),[0]])
    warp=np.clip(np.sort(anch+offs),0,y_ac.shape[1]/sr)
    segs=[]
    for i in range(len(anch)-1):
        s0=int(anch[i]*sr); s1=int(anch[i+1]*sr)
        seg=y_ac[:,s0:s1]; ol=seg.shape[1]
        wl=max(1,int((warp[i+1]-warp[i])*sr))
        segs.append(np.stack([resample(seg[c],wl) for c in range(seg.shape[0])]) if ol>1 and wl!=ol else seg)
    y_ac=np.concatenate(segs,axis=1).astype(np.float32)
    tgt=y.shape[1]
    y_ac=y_ac[:,:tgt] if y_ac.shape[1]>tgt else np.concatenate([y_ac,np.zeros((y_ac.shape[0],tgt-y_ac.shape[1]),dtype=np.float32)],axis=1)
    ac_log.append(f'✓ Timing humanisation  ±{AUTO_CFG["humanise_timing"]*8:.1f} ms  {len(beat_times)} beats warped')
else: ac_log.append('– Timing humanisation  skipped')

# 2d Stereo Correction
if AUTO_CFG['stereo_correct'] > 0 and results['Stereo Width']['raw'] > 0.50:
    ms_r=results['Stereo Width']['raw']; tgt_r=0.45
    sr_=AUTO_CFG['stereo_correct']*min(1,(ms_r-tgt_r)/0.5)
    m=(y_ac[0]+y_ac[1])*0.5; s=(y_ac[0]-y_ac[1])*0.5*(1-sr_*0.45)
    y_ac[0]=(m+s).astype(np.float32); y_ac[1]=(m-s).astype(np.float32)
    ac_log.append(f'✓ Stereo correction    {ms_r:.3f}→~{tgt_r:.2f} M/S')
else: ac_log.append('– Stereo correction    skipped')

# 2e HF Shelf Restore
if AUTO_CFG['hf_restore_db'] > 0 and results['HF Energy']['raw'] < 0.004:
    glin=10**(AUTO_CFG['hf_restore_db']/20)
    b_hf,a_hf=scipy_signal.butter(2,min(12000/(sr/2),0.99),btype='high')
    for ch in range(y_ac.shape[0]):
        y_ac[ch]+=(scipy_signal.filtfilt(b_hf,a_hf,y_ac[ch])*(glin-1)).astype(np.float32)
    ac_log.append(f'✓ HF shelf restore     +{AUTO_CFG["hf_restore_db"]:.1f} dB above 12 kHz')
else: ac_log.append('– HF shelf restore     skipped')

# 2f Noise Floor
if AUTO_CFG['noise_floor_db'] != 0.0:
    nlvl=10**(AUTO_CFG['noise_floor_db']/20)
    wh=np.random.randn(y_ac.shape[1]).astype(np.float32)
    bp=[0.049922035,-0.095993537,0.050612699,-0.004408786]
    ap=[1,-2.494956002,2.017265875,-0.522189400]
    pk=scipy_signal.lfilter(bp,ap,wh).astype(np.float32)
    pk=pk/(np.max(np.abs(pk))+1e-10)*nlvl
    for ch in range(y_ac.shape[0]): y_ac[ch]+=pk
    ac_log.append(f'✓ Noise floor inject   {AUTO_CFG["noise_floor_db"]} dBFS pink noise')
else: ac_log.append('– Noise floor inject   skipped')

# 2g Normalise
pk_ac=np.max(np.abs(y_ac))
if pk_ac>0: y_ac=(y_ac*(10**(AUTO_CFG['output_target']/20)/pk_ac)).astype(np.float32)
ac_log.append(f'✓ Normalised           {AUTO_CFG["output_target"]} dBFS peak')

# ── Step 3: Before/After metrics ─────────────────────────────────────
y_ac_mono=librosa.to_mono(y_ac)
pk2=np.max(np.abs(y_ac_mono)); rms2=np.sqrt(np.mean(y_ac_mono**2))
crest2=20*np.log10(pk2/rms2) if rms2>0 else 20
m2=(y_ac[0]+y_ac[1])*0.5; s2=(y_ac[0]-y_ac[1])*0.5
ms2=np.mean(s2**2)/(np.mean(m2**2)+1e-10)
S2=np.abs(librosa.stft(y_ac_mono)); f2=librosa.fft_frequencies(sr=sr)
hf2=float(np.mean(S2[f2>15000,:]**2)/(np.mean(S2**2)+1e-10))
fl2=float(np.mean(librosa.feature.spectral_flatness(y=y_ac_mono)))

print('  Pipeline stages:')
for l in ac_log: print(f'   {l}')
print()
print('  Before → After key metrics:')
print(f'  {"Metric":<22} {"Before":>10}  {"After":>10}   Change')
print('  '+'─'*58)

def chg(b,a,fmt='.2f',hi=True):
    d=a-b; ok=(d>0)==hi
    return f'{d:+{fmt}}  {("▲" if d>0 else "▼")+" ✓" if ok else ("▲" if d>0 else "▼")+" !"}'

print(f'  {"Dynamic Range (dB)":<22} {results["Dynamic Range"]["raw"]:>10.1f}  {crest2:>10.1f}   {chg(results["Dynamic Range"]["raw"],crest2,".1f",True)}')
print(f'  {"Stereo M/S":<22} {results["Stereo Width"]["raw"]:>10.3f}  {ms2:>10.3f}   {chg(results["Stereo Width"]["raw"],ms2,".3f",False)}')
print(f'  {"HF Energy (%)":<22} {results["HF Energy"]["raw"]*100:>10.3f}  {hf2*100:>10.3f}   {chg(results["HF Energy"]["raw"]*100,hf2*100,".3f",True)}')
print(f'  {"Spectral Flatness":<22} {results["Spectral Flatness"]["raw"]:>10.4f}  {fl2:>10.4f}   {chg(results["Spectral Flatness"]["raw"],fl2,".4f",False)}')
print()

# ── Step 4: Export ────────────────────────────────────────────────────
out_ac=f'{os.path.splitext(fname)[0]}_autocorrected.wav'
sf.write(out_ac, y_ac.T, sr, subtype='PCM_24')
print(f'  ✅ Exported: {out_ac}')
print(f'     WAV 24-bit · {sr} Hz · stereo · {y_ac.shape[1]/sr:.2f}s')
print()
print('═'*58)
print('  ⬇️  Downloading autocorrected file...')
print('═'*58)
files.download(out_ac)

In [ ]:
# ── Cell 6 (Manual): Processing Configuration ────────────────────────
# Use this to override specific stages after running the Autocorrect above,
# or to run the manual pipeline instead. Set any value to 0.0 to skip.

CFG = {
    'dr_expansion'       : 1.0,   # 0=off, 1=subtle, 2=aggressive
    'spectral_smooth'    : 1.0,   # 0=off, 1=light, 3=heavy
    'humanise_timing'    : 0.8,   # 0=off, 1=±4ms, 2=±12ms
    'stereo_correct'     : 0.7,   # 0=off, 1=full
    'hf_restore_db'      : 2.0,   # dB boost above 12kHz, 0=off
    'noise_floor_db'     : -90.0, # dBFS pink noise, 0=off
    'output_lufs_target' : -1.0,  # dBFS peak output
}

print('✅ Manual configuration loaded:')
for k,v in CFG.items(): print(f'   {k:<24} = {v}')

In [ ]:
# ── Cell 7 (Manual): Processing Pipeline ────────────────────────────
from scipy.signal import resample as sci_resample
y_proc = y.copy().astype(np.float32)
log = []

if CFG['dr_expansion']>0 and results['Dynamic Range']['raw']<14:
    rms_p=np.sqrt(np.mean(y_proc[0]**2)); thr=rms_p*0.35
    for ch in range(y_proc.shape[0]):
        mask=np.abs(y_proc[ch])<thr
        y_proc[ch][mask]*=1-(CFG['dr_expansion']*0.12*(1-np.abs(y_proc[ch][mask])/(thr+1e-10)))
    log.append(f'✓ DR expansion')
else: log.append('– DR expansion skipped')

if CFG['spectral_smooth']>0:
    for ch in range(y_proc.shape[0]):
        y_proc[ch]=nr.reduce_noise(y=y_proc[ch],y_noise=y_proc[ch][:sr],sr=sr,
            prop_decrease=min(0.7,0.25*CFG['spectral_smooth']),
            freq_mask_smooth_hz=500,time_mask_smooth_ms=100).astype(np.float32)
    log.append('✓ Spectral smoothing')
else: log.append('– Spectral smoothing skipped')

if CFG['stereo_correct']>0 and results['Stereo Width']['raw']>0.50:
    ms_r=results['Stereo Width']['raw']; sr_=CFG['stereo_correct']*min(1,(ms_r-0.45)/0.5)
    m=(y_proc[0]+y_proc[1])*0.5; s=(y_proc[0]-y_proc[1])*0.5*(1-sr_*0.45)
    y_proc[0]=(m+s).astype(np.float32); y_proc[1]=(m-s).astype(np.float32)
    log.append('✓ Stereo correction')
else: log.append('– Stereo correction skipped')

if CFG['hf_restore_db']>0 and results['HF Energy']['raw']<0.004:
    glin=10**(CFG['hf_restore_db']/20)
    b_hf,a_hf=scipy_signal.butter(2,min(12000/(sr/2),0.99),btype='high')
    for ch in range(y_proc.shape[0]):
        y_proc[ch]+=(scipy_signal.filtfilt(b_hf,a_hf,y_proc[ch])*(glin-1)).astype(np.float32)
    log.append(f'✓ HF restore +{CFG["hf_restore_db"]} dB')
else: log.append('– HF restore skipped')

if CFG['noise_floor_db']!=0.0:
    nlvl=10**(CFG['noise_floor_db']/20)
    wh=np.random.randn(y_proc.shape[1]).astype(np.float32)
    pk=scipy_signal.lfilter([0.049922035,-0.095993537,0.050612699,-0.004408786],
                             [1,-2.494956002,2.017265875,-0.522189400],wh).astype(np.float32)
    pk=pk/(np.max(np.abs(pk))+1e-10)*nlvl
    for ch in range(y_proc.shape[0]): y_proc[ch]+=pk
    log.append('✓ Noise floor injection')
else: log.append('– Noise floor injection skipped')

pk_p=np.max(np.abs(y_proc))
if pk_p>0: y_proc=(y_proc*(10**(CFG['output_lufs_target']/20)/pk_p)).astype(np.float32)
log.append(f'✓ Normalised to {CFG["output_lufs_target"]} dBFS')

print('\n🎛️ Manual processing complete:')
for l in log: print(f'   {l}')

In [ ]:
# ── Cell 8: Before / After Comparison ──────────────────────────────
# Use y_proc (manual) or y_ac (autocorrect) — change the variable below
y_out = y_ac   # ← change to y_proc to compare manual output instead

y_orig_m=librosa.to_mono(y); y_out_m=librosa.to_mono(y_out)
fig,axes=plt.subplots(2,2,figsize=(16,8),facecolor='#07070c')
fig.suptitle('Before vs After Processing',color='#f59e0b',fontsize=13,fontweight='bold')
titles=[('Original Waveform','#f59e0b'),('Processed Waveform','#4ade80'),
        ('Original Spectrogram','#f59e0b'),('Processed Spectrogram','#4ade80')]
for ax,(title,col) in zip(axes.flat,titles):
    ax.set_facecolor('#111118')
    for sp in ax.spines.values(): sp.set_color('#22223a')
    ax.tick_params(colors='#6868a0',labelsize=8)
    ax.set_title(title,color=col,fontsize=10)
t_ax=np.linspace(0,duration,len(y_orig_m))
axes[0,0].plot(t_ax,y_orig_m,color='#f59e0b',lw=0.3,alpha=0.9)
axes[0,1].plot(t_ax[:len(y_out_m)],y_out_m,color='#4ade80',lw=0.3,alpha=0.9)
Do=librosa.amplitude_to_db(np.abs(librosa.stft(y_orig_m)),ref=np.max)
Dp=librosa.amplitude_to_db(np.abs(librosa.stft(y_out_m)),ref=np.max)
librosa.display.specshow(Do,sr=sr,x_axis='time',y_axis='hz',ax=axes[1,0],cmap='magma')
librosa.display.specshow(Dp,sr=sr,x_axis='time',y_axis='hz',ax=axes[1,1],cmap='viridis')
plt.tight_layout()
plt.savefig('beatlab_before_after.png',dpi=150,bbox_inches='tight',facecolor='#07070c')
plt.show()
print('📊 Comparison saved.')

In [ ]:
# ── Cell 9: Export Manual-Processed Audio ───────────────────────────
out_m=f'{os.path.splitext(fname)[0]}_processed.wav'
sf.write(out_m, y_proc.T, sr, subtype='PCM_24')
print(f'✅ Exported: {out_m}  ({sr} Hz · 24-bit · {y_proc.shape[1]/sr:.2f}s)')
files.download(out_m)
files.download('beatlab_ai_analysis.png')
files.download('beatlab_before_after.png')

---
## 📖 Metric & Autocorrect Reference
| Metric | AI signature | Human signature | Autocorrect action |
|---|---|---|---|
| **Dynamic Range** | < 8 dB crest (over-limited) | 12–20 dB | Upward expansion of sub-threshold samples |
| **Spectral Flatness** | > 0.25 (flat) or < 0.03 (peaky) | 0.03–0.20 | Noise reduction to remove anomalous resonances |
| **Timing Regularity** | < 3 ms jitter (machine-perfect) | 5–25 ms variance | Piecewise time-warp with random beat offsets |
| **Stereo Width** | M/S ratio > 0.75 (too wide) | 0.15–0.60 | Side channel attenuation toward 0.45 target |
| **HF Energy** | < 0.002% (dead HF) | 0.2–3% | High-shelf boost above 12 kHz |
| **Noise Floor** | Below −90 dBFS (silent) | −75 to −60 dBFS | Pink noise injection at −90 dBFS |

### Scoring thresholds
- **0–38%** → Likely human-produced
- **38–65%** → Uncertain (may be heavily processed human audio)
- **65–100%** → Likely AI-generated

### Autocorrect strength formula
Each correction strength is computed as: `strength = (score − trigger) / (1 − trigger) × max_strength`  
where *trigger* is the minimum score needed to activate that stage. Metrics scoring below their trigger are left completely untouched.
